# Dashboard Extracts  ·  v3

**Inputs:** `rsrp_clean.parquet` (Notebook 01) · `traffic_clean.parquet` (Notebook 02)
**Outputs:** 8 CSV files in `Data/processed/`, ready to load into Tableau

---

## Why only eight files

v2 produced five extracts. A working draft of this dashboard produced twenty-three. Eight is
the number that falls out of one rule:

> **One file per grain.** Two tables belong together if — and only if — they answer questions
> at the same level of detail.

Everything else is a `WHERE` clause, and Tableau does `WHERE` clauses perfectly well. A
separate file for "weekdays" and another for "weekends" is not a different grain; it is the
same grain filtered twice, and it guarantees the two will drift apart the first time
something is fixed in one and not the other.

### Fact tables — one per grain

| # | File | Grain | Answers |
|---|---|---|---|
| 1 | `fact_rsrp_cells.csv` | bin size × date × hour × operator × cell | every map, every coverage chart |
| 2 | `fact_traffic_bins.csv` | bin size × date × hour × operator × direction × cell | every traffic chart, the coverage↔traffic scatter |
| 3 | `fact_device_totals.csv` | device × operator | handset comparison, the gap table |
| 4 | `fact_rsrp_values.csv` | date × hour × operator × flag × RSRP value | exact distributions and percentiles |
| 5 | `fact_device_values.csv` | device × operator × RSRP value | the quantisation evidence |
| 6 | `fact_tech_mix.csv` | operator × generation | LTE attach rate *(written by Notebook 01)* |

### Dimensions — one workbook, five sheets

| Sheet | Rows | Shared by |
|---|---|---|
| `dim_operator` | 3 | every fact |
| `dim_date` | 3 | RSRP cells, traffic, RSRP values |
| `dim_hour` | 24 | RSRP cells, traffic, RSRP values |
| `dim_cell` | 7,252 | RSRP cells, traffic — **this is the one that makes the scatter work** |
| `dim_device` | 221 | device totals, device values |

### Two files that are gone from v3's first draft

- **`operator_summary.csv`** — every one of its 33 columns is derivable from the facts once
  they share dimensions. §6 proves it column by column rather than asserting it.
- **`device_rsrp.csv`** — device × operator × date. A real grain, but no chart in the
  dashboard asks a question at it. A grain nobody queries does not need a file.

---

## The two rules that make this work

**Rule 1 — export sums and counts, never means.**

A mean cannot be re-aggregated. If a file stores `rsrp_mean` per cell and the dashboard
averages those means across fifty cells, every cell counts equally — a cell with three
samples pulls as hard as one with nine thousand. The answer is wrong, and it is wrong
*silently*, which is worse.

So every extract carries `n_samples` and `rsrp_sum`. Tableau computes
`SUM([rsrp_sum]) / SUM([n_samples])`, which is the true weighted mean at whatever level the
user has filtered to. Means that *are* included (`rsrp_mean`, `tv_median`) are convenience
columns valid **only at the file's own grain** and are named so that is visible.

**Rule 2 — every measure ships in both toggle states.**

The dashboard has two switches:

- *Correct handset reporting artefacts* — exclude the quantised handsets found in §B.5 of
  Notebook 01
- *Suppress implausible traffic samples* — exclude the cumulative-counter records found in
  §C of Notebook 02

Neither can be a filter applied here, because then only one state would exist. Instead each
measure appears twice: `rsrp_sum` (corrected) beside `rsrp_sum_raw` (as delivered), `tv_sum`
beside `tv_sum_raw`. The toggle becomes a Tableau parameter choosing between two columns —
no reload, no second data source, and the two states are guaranteed to come from the same
run of the same pipeline.

---
## Section 0 — Setup

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

try:
    display
except NameError:
    def display(x): print(x.to_string() if hasattr(x, "to_string") else x)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

BASE_DIR    = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
DATA_DIR    = BASE_DIR / "Data"
INTERIM_DIR   = DATA_DIR / "interim"     # Notebooks 01/02 wrote these; nothing else reads them
PROCESSED_DIR = DATA_DIR / "processed"   # the nine files Tableau loads
REF_DIR       = DATA_DIR / "reference"   # audit trail — documented, never a data source
for _d in (PROCESSED_DIR, REF_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Constants — must match Notebooks 01 and 02 exactly
# ─────────────────────────────────────────────────────────────────────────────
BIN_SIZES   = [(0.005, "500m"), (0.01, "1km"), (0.02, "2km")]
JOIN_BIN    = "1km"       # grid used for weak-cell counting and handset matching
CORR_BIN    = "2km"       # grid used for the coverage-to-traffic relationship (see §5)

BAND_LABELS = ["1 Excellent", "2 Good", "3 Fair", "4 Poor", "5 Very Poor"]
BAND_SHORT  = {"1 Excellent": "excellent", "2 Good": "good", "3 Fair": "fair",
               "4 Poor": "poor", "5 Very Poor": "very_poor"}

WEAK_CELL_DBM   = -100    # a cell whose weighted mean RSRP falls below this is "weak"
SERVICE_DBM     = -95     # the service threshold used as the diverging colour pivot
MIN_CELL_SAMPLES = 15     # a cell needs this many samples before it is drawn on a map
MIN_DEVICE_SAMPLES = 200  # a handset needs this many before it is ranked
MIN_MATCH_SAMPLES  = 20   # per operator, per cell, for a location-matched comparison

manifest = []             # (filename, rows, cols, grain) — printed at the end

print(f"Output dir : {PROCESSED_DIR}")

Output dir : /root/v5/Data/processed


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the intermediates. Notebooks 01/02 write parquet when pyarrow is present
# and pickle otherwise; accept either.
# ─────────────────────────────────────────────────────────────────────────────
def load_intermediate(stem):
    pq, pk = INTERIM_DIR / f"{stem}.parquet", INTERIM_DIR / f"{stem}.pkl"
    if pq.exists():
        return pd.read_parquet(pq)
    if pk.exists():
        return pd.read_pickle(pk)
    raise FileNotFoundError(
        f"Neither {pq.name} nor {pk.name} found in {INTERIM_DIR}. "
        f"Run Notebooks 01 and 02 first."
    )


rsrp    = load_intermediate("rsrp_clean")
traffic = load_intermediate("traffic_clean")

# `date` round-trips as an object column; normalise to a plain ISO string so the
# CSVs are unambiguous and Tableau parses them without a locale guess.
for df in (rsrp, traffic):
    df["date"] = df["date"].astype(str)

print(f"RSRP    : {len(rsrp):,} rows  |  flagged (quantised): {int(rsrp['is_quantised'].sum()):,}")
print(f"Traffic : {len(traffic):,} rows  |  flagged (odometer) : {int(traffic['is_odometer'].sum()):,}")

RSRP    : 1,975,572 rows  |  flagged (quantised): 273,112
Traffic : 102,497 rows  |  flagged (odometer) : 14


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def bin_centroid(key_series, deg):
    # Recover a bin's centroid from its key. Both notebooks build the key as
    # round(lat/deg)_round(lon/deg), so this inverts it exactly — which matters
    # more than convenience: it guarantees the RSRP and traffic centroids agree
    # to the last bit, so the join in Section 5 cannot silently half-match.
    parts = key_series.str.split("_", n=1, expand=True)
    # round() clears binary-float residue such as 46.980000000000004, which would
    # otherwise cost ~2 MB of CSV and make two identical centroids compare unequal.
    return ((parts[0].astype(int) * deg).round(6), (parts[1].astype(int) * deg).round(6))


def p10(s):  return s.quantile(0.10)
def p90(s):  return s.quantile(0.90)


def register(name, frame, grain):
    # Write a CSV into the model folder and record it in the manifest.
    path = PROCESSED_DIR / name
    frame.to_csv(path, index=False)
    mb = path.stat().st_size / 1024**2
    manifest.append({"file": name, "rows": len(frame), "cols": frame.shape[1],
                     "MB": round(mb, 2), "grain": grain})
    print(f"Written: {name:<28} {len(frame):>8,} rows x {frame.shape[1]:>2} cols   {mb:>6.2f} MB")
    return path

---
## Section 1 — `rsrp_cells.csv`  ·  the spatial backbone

**Grain:** bin size × date × hour × operator × cell

This one table drives every map and every coverage chart in the dashboard. Because it keeps
`date` and `hour` as columns rather than collapsing them, a single date filter or hour
filter in Tableau re-aggregates it correctly — there is no need for a separate daily file, an
hourly file, or a weekday/weekend file.

**All three bin sizes live in one file**, distinguished by the `bin_size` column, so a grid
selector is a filter rather than a data-source swap.

### What is stored, and why

| Column group | Re-aggregates? | Note |
|---|---|---|
| `n_samples`, `rsrp_sum` | ✅ | the pair that gives a correct weighted mean at any level |
| `rsrp_min`, `rsrp_max` | ✅ | MIN of MIN, MAX of MAX |
| `n_excellent` … `n_very_poor` | ✅ | band shares recompute as `SUM(n_band)/SUM(n_samples)` |
| `rsrp_mean_here`, `rsrp_median_here` | ❌ | valid **only** at this row's grain — the `_here` suffix is the warning |

Every one of those exists twice: plain (quantised handsets excluded) and `_raw`
(as delivered). The `_here` columns are the exception — only the corrected version is
provided, because a non-aggregatable convenience column in two flavours is two ways to be
wrong rather than one.

In [4]:
GRP = ["date", "hour", "RadioOperatorName"]

band_cols = {}   # filled below, used again in Section 2


def cells_for_binsize(df, deg, label):
    key = f"bin_{label}"
    g   = GRP + [key]

    # ── Band counts: one column per coverage band, pivoted wide ──────────────
    def band_counts(frame, suffix):
        wide = (frame.groupby(g + ["coverage_band"], observed=True)
                     .size().unstack("coverage_band", fill_value=0))
        wide = wide.reindex(columns=BAND_LABELS, fill_value=0)
        wide.columns = [f"n_{BAND_SHORT[c]}{suffix}" for c in wide.columns]
        return wide

    # ── RAW: every row, quantised handsets included ─────────────────────────
    raw = (df.groupby(g, observed=True)
             .agg(n_samples_raw=("RSRP", "size"),
                  rsrp_sum_raw =("RSRP", "sum"),
                  rsrp_min_raw =("RSRP", "min"),
                  rsrp_max_raw =("RSRP", "max")))
    raw = raw.join(band_counts(df, "_raw"))

    # ── ADJUSTED: quantised handsets excluded ───────────────────────────────
    adj_src = df[~df["is_quantised"]]
    adj = (adj_src.groupby(g, observed=True)
                  .agg(n_samples   =("RSRP", "size"),
                       rsrp_sum    =("RSRP", "sum"),
                       rsrp_min    =("RSRP", "min"),
                       rsrp_max    =("RSRP", "max"),
                       rsrp_median_here=("RSRP", "median")))
    adj = adj.join(band_counts(adj_src, ""))

    # A cell can exist in raw but not in adj — every one of its samples came
    # from a flagged handset. Left join keeps it, with zeros on the adjusted
    # side, so the map does not silently lose coverage when the toggle flips.
    out = raw.join(adj, how="left").reset_index()

    count_like = [c for c in out.columns if c.startswith("n_")]
    out[count_like] = out[count_like].fillna(0).astype(int)
    out["rsrp_sum"] = out["rsrp_sum"].fillna(0).astype("int64")

    # RSRP is integral. Nullable Int64 keeps it that way for the cells where the
    # adjusted side is empty, instead of demoting the column to float and writing
    # "-83.0" where the source says -83.
    for c in ["rsrp_min", "rsrp_max", "rsrp_min_raw", "rsrp_max_raw"]:
        out[c] = out[c].astype("Int64")

    out["bin_size"] = label
    out = out.rename(columns={key: "cell_key"})
    out["lat"], out["lon"] = bin_centroid(out["cell_key"], deg)

    # Convenience means — valid at THIS grain only, hence the suffix
    out["rsrp_mean_here"] = np.where(
        out["n_samples"] > 0, out["rsrp_sum"] / out["n_samples"].replace(0, np.nan), np.nan
    ).round(4)
    out["rsrp_median_here"] = out["rsrp_median_here"].round(2)

    return out


rsrp_cells = pd.concat(
    [cells_for_binsize(rsrp, deg, label) for deg, label in BIN_SIZES],
    ignore_index=True,
)

ordered = (["bin_size", "cell_key", "lat", "lon"]
           + ["date", "hour", "RadioOperatorName",
              "n_samples", "rsrp_sum", "rsrp_min", "rsrp_max",
              "rsrp_mean_here", "rsrp_median_here"]
           + [f"n_{BAND_SHORT[b]}" for b in BAND_LABELS]
           + ["n_samples_raw", "rsrp_sum_raw", "rsrp_min_raw", "rsrp_max_raw"]
           + [f"n_{BAND_SHORT[b]}_raw" for b in BAND_LABELS])
rsrp_cells = rsrp_cells[ordered]

print(f"rsrp_cells: {len(rsrp_cells):,} rows")
print(rsrp_cells.groupby("bin_size", observed=True).size().to_string())
display(rsrp_cells.head(3))

rsrp_cells: 87,802 rows
bin_size
1km     27425
2km     16602
500m    43775


,bin_size,cell_key,lat,lon,date,hour,RadioOperatorName,n_samples,rsrp_sum,rsrp_min,rsrp_max,rsrp_mean_here,rsrp_median_here,n_excellent,n_good,n_fair,n_poor,n_very_poor,n_samples_raw,rsrp_sum_raw,rsrp_min_raw,rsrp_max_raw,n_excellent_raw,n_good_raw,n_fair_raw,n_poor_raw,n_very_poor_raw
0,500m,4890_9399,24.450,46.995,2019-11-02,0,Operator A,1,-83,-83,-83,-83.0,-83.0,0,1,0,0,0,1,-83,-83,-83,0,1,0,0,0
1,500m,4901_9326,24.505,46.630,2019-11-02,0,Operator A,1,-107,-107,-107,-107.0,-107.0,0,0,0,1,0,1,-107,-107,-107,0,0,0,1,0
2,500m,4904_9333,24.520,46.665,2019-11-02,0,Operator A,2,-207,-105,-102,-103.5,-103.5,0,0,0,2,0,2,-207,-105,-102,0,0,0,2,0


### Validation — the counts must reconcile

Three independent checks, each of which would catch a different class of mistake:

1. **Band counts sum to the sample count.** Catches a row that fell outside every band.
2. **Sample counts sum to the source table.** Catches rows lost in a join or a `fillna`.
3. **The weighted mean recovered from the sums matches the mean of the source.** Catches
   a sum computed over the wrong subset — the failure that a row-count check cannot see.

In [5]:
band_adj = [f"n_{BAND_SHORT[b]}" for b in BAND_LABELS]
band_raw = [f"n_{BAND_SHORT[b]}_raw" for b in BAND_LABELS]

assert (rsrp_cells[band_adj].sum(axis=1) == rsrp_cells["n_samples"]).all(), \
    "Adjusted band counts do not sum to n_samples"
assert (rsrp_cells[band_raw].sum(axis=1) == rsrp_cells["n_samples_raw"]).all(), \
    "Raw band counts do not sum to n_samples_raw"

for deg, label in BIN_SIZES:
    sub = rsrp_cells[rsrp_cells["bin_size"] == label]
    assert sub["n_samples_raw"].sum() == len(rsrp), \
        f"{label}: raw samples {sub['n_samples_raw'].sum():,} != source {len(rsrp):,}"
    assert sub["n_samples"].sum() == int((~rsrp["is_quantised"]).sum()), \
        f"{label}: adjusted samples do not match the unflagged source rows"

    mean_from_extract = sub["rsrp_sum"].sum() / sub["n_samples"].sum()
    mean_from_source  = rsrp.loc[~rsrp["is_quantised"], "RSRP"].mean()
    assert abs(mean_from_extract - mean_from_source) < 1e-9, \
        f"{label}: weighted mean {mean_from_extract:.6f} != source {mean_from_source:.6f}"

print("All three reconciliation checks passed, for all three bin sizes.")
print(f"  rows in source            : {len(rsrp):,}")
print(f"  unflagged rows in source  : {int((~rsrp['is_quantised']).sum()):,}")
print(f"  weighted mean (corrected) : {mean_from_extract:.4f} dBm")
print(f"  weighted mean (as deliv.) : {rsrp['RSRP'].mean():.4f} dBm")

All three reconciliation checks passed, for all three bin sizes.
  rows in source            : 1,975,572
  unflagged rows in source  : 1,702,460
  weighted mean (corrected) : -83.1587 dBm
  weighted mean (as deliv.) : -84.2380 dBm


In [6]:
print("Held for Section 6 — columns are renamed to the v2 convention before writing.")

Held for Section 6 — columns are renamed to the v2 convention before writing.


---
## Section 2 — `traffic_bins.csv`

**Grain:** bin size × date × hour × operator × direction × cell

Same shape as `rsrp_cells`, same three bin sizes in one file, built from the same key
construction — which is what allows the two to join on `cell_key` in Section 5 without a
spatial operation.

`tv_sum` excludes the odometer records; `tv_sum_raw` includes them. `tv_median_here` is a
convenience column at this row's grain only: a median cannot be re-aggregated, and 14 rows
cannot move a median anyway, so it exists in one flavour.

In [7]:
TGRP = ["date", "hour", "RadioOperatorName", "TrafficDirection"]


def traffic_for_binsize(df, deg, label):
    key = f"bin_{label}"
    g   = TGRP + [key]

    raw = (df.groupby(g, observed=True)
             .agg(n_samples_raw=("TrafficVolume", "size"),
                  tv_sum_raw   =("TrafficVolume", "sum"),
                  tv_max_raw   =("TrafficVolume", "max")))

    cln_src = df[~df["is_odometer"]]
    cln = (cln_src.groupby(g, observed=True)
                  .agg(n_samples     =("TrafficVolume", "size"),
                       tv_sum        =("TrafficVolume", "sum"),
                       tv_max        =("TrafficVolume", "max"),
                       tv_median_here=("TrafficVolume", "median")))

    out = raw.join(cln, how="left").reset_index()
    out["n_samples"] = out["n_samples"].fillna(0).astype(int)
    out["tv_sum"]    = out["tv_sum"].fillna(0.0)

    out["bin_size"] = label
    out = out.rename(columns={key: "cell_key"})
    out["lat"], out["lon"] = bin_centroid(out["cell_key"], deg)

    # 5 dp = 10 bytes of traffic. Below any plausible measurement resolution, and it
    # keeps the file inside the size a single upload will accept.
    for c in ["tv_sum", "tv_sum_raw", "tv_max", "tv_max_raw", "tv_median_here"]:
        out[c] = out[c].round(5)
    return out


traffic_bins = pd.concat(
    [traffic_for_binsize(traffic, deg, label) for deg, label in BIN_SIZES],
    ignore_index=True,
)

ordered = (["bin_size", "cell_key", "lat", "lon"]
           + ["date", "hour", "RadioOperatorName", "TrafficDirection",
              "n_samples", "tv_sum", "tv_max", "tv_median_here",
              "n_samples_raw", "tv_sum_raw", "tv_max_raw"])
traffic_bins = traffic_bins[ordered]

# Reconciliation
for deg, label in BIN_SIZES:
    sub = traffic_bins[traffic_bins["bin_size"] == label]
    assert sub["n_samples_raw"].sum() == len(traffic), f"{label}: traffic row count mismatch"
    assert abs(sub["tv_sum_raw"].sum() - traffic["TrafficVolume"].sum()) < 1e-3, \
        f"{label}: raw volume does not reconcile"
    assert abs(sub["tv_sum"].sum()
               - traffic.loc[~traffic["is_odometer"], "TrafficVolume"].sum()) < 1e-3, \
        f"{label}: clean volume does not reconcile"

print(f"traffic_bins: {len(traffic_bins):,} rows")
print(traffic_bins.groupby("bin_size", observed=True).size().to_string())
print(f"\nTotal volume as delivered : {traffic['TrafficVolume'].sum()/1024:>10,.1f} GB")
print(f"Total volume corrected    : "
      f"{traffic.loc[~traffic['is_odometer'],'TrafficVolume'].sum()/1024:>10,.1f} GB")
print("Reconciliation passed for all three bin sizes.")
display(traffic_bins.head(3))

traffic_bins: 146,284 rows
bin_size
1km     50265
2km     38505
500m    57514

Total volume as delivered :    3,004.2 GB
Total volume corrected    :    1,602.6 GB
Reconciliation passed for all three bin sizes.


,bin_size,cell_key,lat,lon,date,hour,RadioOperatorName,TrafficDirection,n_samples,tv_sum,tv_max,tv_median_here,n_samples_raw,tv_sum_raw,tv_max_raw
0,500m,4892_9396,24.46,46.980,2019-11-02,0,Operator A,Downlink,1,4.25864,4.25864,4.25864,1,4.25864,4.25864
1,500m,4898_9326,24.49,46.630,2019-11-02,0,Operator A,Downlink,2,136.69485,89.77302,68.34742,2,136.69485,89.77302
2,500m,4898_9385,24.49,46.925,2019-11-02,0,Operator A,Downlink,2,16.62169,16.44021,8.31085,2,16.62169,16.44021


In [8]:
print("Held for Section 6.")

Held for Section 6.


---
## Section 3 — the handset tables

One file, one grain: **`fact_device_totals.csv`** — device × operator, across all three
days. This is the comparison table: how does one handset model fare on Operator A versus
Operator B?

### The trap this section is built to avoid

A naive handset comparison computes a model's mean RSRP on each operator and reports the
difference. In this dataset that produces confident, arithmetically correct, and **causally
meaningless** numbers.

The worked example: `Nokia 5.1 Plus` reads 18.1 dB worse on one operator than the other. But
its Operator A samples are 104 readings from a single cell, at night; its Operator B samples
are 247 readings from two different cells, in the afternoon. **Zero cells in common.** That
is a comparison of two neighbourhoods, not two networks.

So the file carries two gap measures:

- `gap_vs_best` — the naive difference. Kept because it is what a reader expects to see.
- `matched_gap_vs_best` — computed only over cells where **both** operators have at least
  20 readings of that model, weighted by sample count. Null when no such cell exists.

`comparable` is True only when the matched comparison is available. A dashboard should make
network claims from the matched column and use the naive one for context.

In [9]:
DGRP = ["DeviceName", "DeviceManufacturer", "RadioOperatorName"]

dev = (rsrp.groupby(DGRP, observed=True)
           .agg(n_samples =("RSRP", "size"),
                rsrp_sum  =("RSRP", "sum"),
                rsrp_mean =("RSRP", "mean"),
                rsrp_median=("RSRP", "median"),
                rsrp_std  =("RSRP", "std"),
                rsrp_min  =("RSRP", "min"),
                rsrp_max  =("RSRP", "max"),
                rsrp_p10  =("RSRP", p10),
                rsrp_p90  =("RSRP", p90))
           .reset_index())
dev = dev[dev["n_samples"] > 0].copy()

# Carry the quantisation flag through: a model/operator pair is flagged or it is not.
qflag = (rsrp.groupby(DGRP, observed=True)["is_quantised"].max()
             .rename("is_quantised").reset_index())
dev = dev.merge(qflag, on=DGRP, how="left")

# `is_solid` gates the ranking: below MIN_DEVICE_SAMPLES the mean is too noisy to
# rank against another operator, so the model is carried but not compared.
dev["is_solid"] = dev["n_samples"] >= MIN_DEVICE_SAMPLES

for c in ["rsrp_mean", "rsrp_median", "rsrp_std", "rsrp_p10", "rsrp_p90"]:
    dev[c] = dev[c].round(3)

print(f"device x operator rows: {len(dev):,}")
print(f"  solid (>= {MIN_DEVICE_SAMPLES} samples): {int(dev['is_solid'].sum()):,}")
print(f"  flagged as quantised           : {int(dev['is_quantised'].sum()):,}")

device x operator rows: 396
  solid (>= 200 samples): 245
  flagged as quantised           : 6


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Naive gap: best operator for each model, among the solid pairs only
# ─────────────────────────────────────────────────────────────────────────────
solid = dev[dev["is_solid"]]

best = (solid.sort_values("rsrp_mean", ascending=False)
             .groupby("DeviceName", observed=True)
             .agg(best_operator=("RadioOperatorName", "first"),
                  best_op_rsrp =("rsrp_mean", "first"),
                  n_operators_solid=("RadioOperatorName", "nunique"))
             .reset_index())

dev = dev.merge(best, on="DeviceName", how="left")
dev["n_operators_solid"] = dev["n_operators_solid"].fillna(0).astype(int)
dev["gap_vs_best"] = np.where(
    dev["is_solid"] & (dev["n_operators_solid"] >= 2),
    (dev["rsrp_mean"] - dev["best_op_rsrp"]).round(3),
    np.nan,
)

print(f"Models observed on >= 2 operators with enough samples: "
      f"{dev.loc[dev['n_operators_solid'] >= 2, 'DeviceName'].nunique():,}")

Models observed on >= 2 operators with enough samples: 65


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Location-matched gap
#
# For each (model, cell) pair where two or more operators each have at least
# MIN_MATCH_SAMPLES readings, compare within the cell, then weight each cell by
# the smaller of the two sample counts — a cell where one side has 500 readings
# and the other has 21 is worth about 21 readings of evidence, not 500.
# ─────────────────────────────────────────────────────────────────────────────
KEY = f"bin_{JOIN_BIN}"

cellwise = (rsrp.groupby(["DeviceName", KEY, "RadioOperatorName"], observed=True)
                .agg(n=("RSRP", "size"), s=("RSRP", "sum"))
                .reset_index())
cellwise = cellwise[cellwise["n"] >= MIN_MATCH_SAMPLES]
cellwise["m"] = cellwise["s"] / cellwise["n"]

# Keep only (model, cell) groups seen on more than one operator
multi = (cellwise.groupby(["DeviceName", KEY], observed=True)["RadioOperatorName"]
                 .transform("nunique") >= 2)
cellwise = cellwise[multi]

# Self-join within (model, cell) to get every operator pair
pairs = cellwise.merge(cellwise, on=["DeviceName", KEY], suffixes=("_a", "_b"))
pairs = pairs[pairs["RadioOperatorName_a"] != pairs["RadioOperatorName_b"]]
pairs["w"]     = np.minimum(pairs["n_a"], pairs["n_b"])
pairs["delta"] = pairs["m_a"] - pairs["m_b"]
pairs["wd"]    = pairs["w"] * pairs["delta"]

matched = (pairs.groupby(["DeviceName", "RadioOperatorName_a", "RadioOperatorName_b"],
                         observed=True)
                .agg(n_matched_cells=(KEY, "nunique"),
                     w_sum=("w", "sum"), wd_sum=("wd", "sum"))
                .reset_index())
matched["matched_delta"] = matched["wd_sum"] / matched["w_sum"]

# For each (model, operator) keep the comparison against its best rival
matched_best = (matched.sort_values("matched_delta")
                       .groupby(["DeviceName", "RadioOperatorName_a"], observed=True)
                       .agg(matched_vs_operator=("RadioOperatorName_b", "first"),
                            matched_gap_vs_best=("matched_delta", "first"),
                            n_matched_cells=("n_matched_cells", "first"),
                            matched_weight=("w_sum", "first"))
                       .reset_index()
                       .rename(columns={"RadioOperatorName_a": "RadioOperatorName"}))

dev = dev.merge(matched_best, on=["DeviceName", "RadioOperatorName"], how="left")
dev["matched_gap_vs_best"] = dev["matched_gap_vs_best"].round(3)
dev["n_matched_cells"] = dev["n_matched_cells"].fillna(0).astype(int)
dev["matched_weight"]  = dev["matched_weight"].fillna(0).astype(int)

# `comparable` = the strict definition. A dashboard makes network claims from here.
dev["comparable"] = dev["n_matched_cells"] > 0

n_cmp = dev.loc[dev["comparable"], "DeviceName"].nunique()
n_nai = dev.loc[dev["n_operators_solid"] >= 2, "DeviceName"].nunique()
print(f"Models with a NAIVE   cross-operator comparison : {n_nai:,}")
print(f"Models with a MATCHED cross-operator comparison : {n_cmp:,}")
print(f"\n{n_nai - n_cmp} models look comparable but share no cell between operators.")
print("Those are exactly the ones that produce confident, meaningless gap numbers.")

Models with a NAIVE   cross-operator comparison : 65
Models with a MATCHED cross-operator comparison : 29

36 models look comparable but share no cell between operators.
Those are exactly the ones that produce confident, meaningless gap numbers.


In [12]:
# Worked example: the naive gap versus the matched gap, for the models where both exist
example = (dev[dev["comparable"] & dev["gap_vs_best"].notna() & (dev["gap_vs_best"] < 0)]
             .nsmallest(8, "gap_vs_best")
             [["DeviceName", "RadioOperatorName", "n_samples",
               "gap_vs_best", "matched_gap_vs_best", "n_matched_cells"]])
print("Naive gap vs location-matched gap (worst 8 with both available):")
display(example)

disagree = (example["gap_vs_best"] - example["matched_gap_vs_best"]).abs()
print(f"\nMedian disagreement between the two measures: {disagree.median():.2f} dB")
print("That difference is geography, not network quality.")

Naive gap vs location-matched gap (worst 8 with both available):


,DeviceName,RadioOperatorName,n_samples,gap_vs_best,matched_gap_vs_best,n_matched_cells
47,Galaxy A20S,Operator B,1324,-17.643,-23.027,8
44,Galaxy A20,Operator B,9395,-12.263,-10.622,3
75,Galaxy A7(2017),Operator C,14127,-11.447,-15.227,2
183,Galaxy S8,Operator C,876,-11.139,-15.239,2
149,Galaxy Note9,Operator A,28561,-11.120,-10.132,14
158,Galaxy S10+,Operator B,3438,-10.603,-16.825,1
188,Galaxy S9,Operator B,316,-10.591,-10.686,1
62,Galaxy A50,Operator B,5008,-9.517,-7.305,4



Median disagreement between the two measures: 3.00 dB
That difference is geography, not network quality.


In [13]:
dev_out = dev[[
    "DeviceName", "DeviceManufacturer", "RadioOperatorName",
    "n_samples", "rsrp_sum",
    "rsrp_mean", "rsrp_median", "rsrp_std", "rsrp_min", "rsrp_max", "rsrp_p10", "rsrp_p90",
    "is_quantised", "is_solid",
    "best_operator", "best_op_rsrp", "n_operators_solid", "gap_vs_best",
    "comparable", "matched_vs_operator", "matched_gap_vs_best",
    "n_matched_cells", "matched_weight",
]].sort_values(["DeviceName", "RadioOperatorName"]).reset_index(drop=True)

assert dev_out["n_samples"].sum() == len(rsrp), "device totals do not reconcile with source"
print(f"Reconciliation passed: {dev_out['n_samples'].sum():,} = {len(rsrp):,}")

print(f"device x operator table ready: {len(dev_out):,} rows")

Reconciliation passed: 1,975,572 = 1,975,572
device x operator table ready: 396 rows


---
## Section 4 — the value-count tables

Two small files that carry the **full distribution** rather than summary statistics.

Storing `(value, count)` pairs instead of the raw rows compresses 1.98 million readings into
a few tens of thousands of rows while losing nothing: any percentile, any histogram, any
mode is recoverable exactly. A `rsrp_median` column, by contrast, is one number that cannot
be recombined with any other.

- **`rsrp_value_counts.csv`** — date × hour × operator × `is_quantised` × RSRP. The
  distribution charts, and exact percentiles at any filter level. The flag is a column, not
  a filter, so the toggle works here too.
- **`device_value_counts.csv`** — device × operator × RSRP. This is the **evidence** for the
  quantisation finding: it is what lets a reader see that `Galaxy Note8` on Operator A puts
  87% of its readings on three values, rather than having to take the claim on trust.

In [14]:
val = (rsrp.groupby(["date", "hour", "RadioOperatorName", "is_quantised", "RSRP"],
                    observed=True)
           .size().rename("n").reset_index()
           .sort_values(["date", "hour", "RadioOperatorName", "is_quantised", "RSRP"])
           .reset_index(drop=True))

assert val["n"].sum() == len(rsrp), "value counts do not reconcile with source"

# Independent check: recover the mean from the value counts alone.
mean_from_counts = (val["RSRP"] * val["n"]).sum() / val["n"].sum()
assert abs(mean_from_counts - rsrp["RSRP"].mean()) < 1e-9, "mean from counts disagrees"
print(f"Reconciliation passed: {val['n'].sum():,} readings, "
      f"mean recovered = {mean_from_counts:.4f} dBm")

print(f"rsrp value counts ready: {len(val):,} rows")

Reconciliation passed: 1,975,572 readings, mean recovered = -84.2380 dBm
rsrp value counts ready: 16,786 rows


In [15]:
dval = (rsrp.groupby(["DeviceName", "RadioOperatorName", "RSRP"], observed=True)
            .size().rename("n").reset_index())
dval = dval.merge(qflag, on=["DeviceName", "RadioOperatorName"], how="left")
dval = (dval.sort_values(["DeviceName", "RadioOperatorName", "RSRP"])
            .reset_index(drop=True))

assert dval["n"].sum() == len(rsrp), "device value counts do not reconcile"

# Show the evidence for the worst flagged pair, straight out of the extract.
worst = (dval[dval["is_quantised"]]
         .groupby(["DeviceName", "RadioOperatorName"], observed=True)["n"].sum().idxmax())
ev = (dval[(dval["DeviceName"] == worst[0]) & (dval["RadioOperatorName"] == worst[1])]
      .nlargest(6, "n"))
tot = ev["n"].sum()
print(f"Evidence for {worst[0]} on {worst[1]} — top values by frequency:\n")
for _, row in ev.iterrows():
    share = 100 * row["n"] / dval[(dval["DeviceName"] == worst[0]) &
                                  (dval["RadioOperatorName"] == worst[1])]["n"].sum()
    print(f"  {row['RSRP']:>5} dBm : {row['n']:>7,}  ({share:>5.1f}%)  "
          + "#" * int(share / 2))

print(f"device value counts ready: {len(dval):,} rows")

Evidence for Galaxy Note8 on Operator A — top values by frequency:

   -107 dBm :  94,439  ( 44.8%)  ######################
    -71 dBm :  47,253  ( 22.4%)  ###########
    -83 dBm :  42,201  ( 20.0%)  ##########
   -104 dBm :   5,807  (  2.8%)  #
    -79 dBm :     859  (  0.4%)  
    -77 dBm :     824  (  0.4%)  
device value counts ready: 14,403 rows


---
## Section 5 — the coverage ↔ traffic relationship

Both spatial extracts are keyed on the same grid, so the relationship is a plain join on
`cell_key` — no spatial operation, no geometry library.

This section produces **no file**. It is a validation of the two extracts: if the
relationship cannot be reproduced from the CSVs alone, the extracts are wrong. It is also
where the single most easily-abused number in this project gets pinned down honestly.

### The number depends on the grid, and that is not a defect

Run the join at 500 m and the correlation is zero. Run it at 2 km and it is a solid +0.35.
Neither is a mistake, and picking whichever one suits an argument would be.

This is the **modifiable areal unit problem** — a well-documented property of any analysis
that aggregates point data into arbitrary cells. Two effects drive it here:

1. **At fine scale, the signal is swamped.** A 500 m cell might hold one person indoors on
   a video call and another outdoors checking email. RSRP differs by 20 dB between them for
   reasons that have nothing to do with the operator's network, and traffic differs by three
   orders of magnitude for reasons that have nothing to do with coverage.
2. **The mechanism itself operates at neighbourhood scale.** Operators dimension capacity by
   district, not by 500 m square. A relationship between coverage and demand, if it exists,
   exists at the scale at which the decisions are made.

So the honest report is a **surface**, not a number: the coefficient across both the grid
size and the minimum-sample floor, so a reader can see where it is stable and where it is
not.

In [16]:
def corr_at(bin_label, floor, direction="Downlink", per_window=False):
    # Coverage side: weighted mean RSRP per cell, from sums (never a mean of means)
    a = (rsrp_cells[rsrp_cells["bin_size"] == bin_label]
         .groupby("cell_key", observed=True)
         .agg(n=("n_samples", "sum"), s=("rsrp_sum", "sum")).reset_index())
    a = a[a["n"] >= floor]
    a["rsrp_mean"] = a["s"] / a["n"]

    # Traffic side
    t = traffic_bins[traffic_bins["bin_size"] == bin_label]
    if direction:
        t = t[t["TrafficDirection"] == direction]
    b = (t.groupby("cell_key", observed=True)
          .agg(tv=("tv_sum", "sum"), tn=("n_samples", "sum")).reset_index())
    b = b[b["tn"] >= floor]

    j = a.merge(b, on="cell_key", how="inner")
    if len(j) < 30:
        return None

    # per_window divides by the number of reporting windows, which removes the
    # "busier cells simply have more records" component of the total.
    y = np.log10(j["tv"] / j["tn"]) if per_window else np.log10(j["tv"])
    r, p   = pearsonr(j["rsrp_mean"], y)
    rs, ps = spearmanr(j["rsrp_mean"], y)
    return {"bin_size": bin_label, "min_samples": floor, "n_cells": len(j),
            "pearson_r": round(r, 3), "spearman_rho": round(rs, 3),
            "p_value": f"{p:.1e}", "sig_at_0.01": p < 0.01}


surface = [corr_at(lbl, f) for _, lbl in BIN_SIZES for f in [1, 15, 30, 50, 100]]
surface_df = pd.DataFrame([x for x in surface if x])

print("Total downlink volume vs mean RSRP, by grid size and sample floor")
display(surface_df.pivot(index="min_samples", columns="bin_size", values="pearson_r"))
print("\nCell counts behind each coefficient:")
display(surface_df.pivot(index="min_samples", columns="bin_size", values="n_cells"))

Total downlink volume vs mean RSRP, by grid size and sample floor


bin_size,1km,2km,500m
min_samples,,,
1,0.262,0.449,0.079
15,0.128,0.343,-0.039
30,-0.014,0.359,-0.035
50,-0.068,0.278,-0.073
100,-0.081,0.152,-0.193



Cell counts behind each coefficient:


bin_size,1km,2km,500m
min_samples,,,
1,1290,459,3223
15,676,313,848
30,468,265,448
50,325,225,219
100,130,161,68


In [17]:
# Robustness: does the 2 km result survive changing the traffic measure and the
# direction? A finding that only holds for one arbitrary choice is not a finding.
variants = []
for direction in ["Downlink", "Uplink", None]:
    for per_window in [False, True]:
        for floor in [15, 30, 50]:
            v = corr_at(CORR_BIN, floor, direction, per_window)
            if v:
                v["direction"] = direction or "both"
                v["measure"]   = "MB per window" if per_window else "total MB"
                variants.append(v)

var_df = pd.DataFrame(variants)[
    ["direction", "measure", "min_samples", "n_cells", "pearson_r", "spearman_rho",
     "p_value", "sig_at_0.01"]
]
display(var_df)

print(f"\nAt {CORR_BIN}: {int(var_df['sig_at_0.01'].sum())} of {len(var_df)} variants are "
      f"significant at p < 0.01, and every coefficient is positive.")
print(f"Range of r: {var_df['pearson_r'].min():+.3f} to {var_df['pearson_r'].max():+.3f}")

,direction,measure,min_samples,n_cells,pearson_r,spearman_rho,p_value,sig_at_0.01
0,Downlink,total MB,15,313,0.343,0.349,4.6e-10,True
1,Downlink,total MB,30,265,0.359,0.332,1.9e-09,True
2,Downlink,total MB,50,225,0.278,0.253,2.3e-05,True
3,Downlink,MB per window,15,313,0.182,0.160,1.3e-03,True
4,Downlink,MB per window,30,265,0.199,0.155,1.1e-03,True
5,Downlink,MB per window,50,225,0.121,0.106,7.0e-02,False
6,Uplink,total MB,15,313,0.346,0.342,3.0e-10,True
7,Uplink,total MB,30,267,0.341,0.330,1.1e-08,True
8,Uplink,total MB,50,227,0.238,0.233,3.0e-04,True
9,Uplink,MB per window,15,313,0.174,0.112,2.0e-03,True



At 2km: 15 of 18 variants are significant at p < 0.01, and every coefficient is positive.
Range of r: +0.063 to +0.400


In [18]:
# ── The headline figure, stated with everything needed to reproduce it ───────
head = corr_at(CORR_BIN, 30, "Downlink", per_window=False)

print("COVERAGE ↔ TRAFFIC — headline result")
print("─" * 62)
print(f"  grid                : {head['bin_size']} cells")
print(f"  minimum samples/cell: {head['min_samples']} (both sides)")
print(f"  traffic measure     : total downlink MB, log10")
print(f"  cells               : {head['n_cells']}")
print(f"  Pearson r           : {head['pearson_r']:+.3f}   (p = {head['p_value']})")
print(f"  Spearman rho        : {head['spearman_rho']:+.3f}")
print()
print("Reading: districts with better average coverage carry more traffic. The effect")
print("is real and stable at neighbourhood scale, and it explains roughly")
print(f"{100*head['pearson_r']**2:.0f}% of the variance — coverage is one input among many.")
print()
print("Two cautions that belong beside this number, not in a footnote:")
print("  1. It is a 2 km result. At 500 m there is no relationship at all.")
print("  2. Causation runs both ways: operators build capacity where demand already is,")
print("     so this cannot be read as 'better coverage causes more usage'.")

COVERAGE ↔ TRAFFIC — headline result
──────────────────────────────────────────────────────────────
  grid                : 2km cells
  minimum samples/cell: 30 (both sides)
  traffic measure     : total downlink MB, log10
  cells               : 265
  Pearson r           : +0.359   (p = 1.9e-09)
  Spearman rho        : +0.332

Reading: districts with better average coverage carry more traffic. The effect
is real and stable at neighbourhood scale, and it explains roughly
13% of the variance — coverage is one input among many.

Two cautions that belong beside this number, not in a footnote:
  1. It is a 2 km result. At 500 m there is no relationship at all.
  2. Causation runs both ways: operators build capacity where demand already is,
     so this cannot be read as 'better coverage causes more usage'.


---
## Section 6 — the v2-compatible export

Everything above computed the numbers. This section decides what they are **called**, and
that decision is worth more care than it looks.

### The naming trap this section exists to avoid

v2 had no artefact correction, so its `n_samples` and `rsrp_mean` meant *as delivered*.
v3 and v4 flipped that — there, `n_samples` was the corrected figure and `n_samples_raw` the
as-delivered one.

If v5 shipped the v3 convention under the v2 filenames, every existing worksheet would keep
running and **silently start showing different numbers**. That is worse than a rename: a
rename fails loudly and gets fixed in a minute; a changed meaning gets believed.

So v5 inverts the suffix instead. **Every column that exists in v2 keeps its v2 meaning**,
and the new state gets a new suffix:

| File | v2 columns mean | New columns |
|---|---|---|
| `rsrp_cells.csv` | **as delivered** — no artefact correction, as in v2 | `*_adj` = quantised handsets excluded |
| `traffic_bins.csv` | **suppressed** — v2 deleted the 14 odometer rows before writing | `*_raw` = those rows included |

The two files carry opposite suffixes because v2 treated the two defects differently. That is
not tidy, but it is true, and the alternative is changing what a column means without saying so.

### The one change that does break something

`bin_size` is new on `rsrp_cells.csv`, and it stacks three grids into one table. An existing
worksheet with no `bin_size` filter will show **three times** the sample count.

The fix is one action, not one per sheet: drag `bin_size` to Filters on any sheet, keep
`1km`, then right-click the filter ▸ **Apply to Worksheets ▸ All Using This Data Source**.
After that every v2-era number matches exactly.

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# rsrp_cells — swap the suffixes so the v2 column names keep the v2 meaning
#
#   v4 name          ->  v5 name        meaning
#   n_samples_raw    ->  n_samples      as delivered   (what v2 called n_samples)
#   n_samples        ->  n_samples_adj  artefact-corrected
# ─────────────────────────────────────────────────────────────────────────────
adj_cols = [c for c in rsrp_cells.columns
            if not c.endswith("_raw") and c.startswith(("n_", "rsrp_")) and c not in ("n_samples_raw",)]
ren = {}
for c in list(rsrp_cells.columns):
    if c.endswith("_raw"):
        ren[c] = c[:-4]                     # n_samples_raw -> n_samples
    elif c.startswith(("n_", "rsrp_")) and not c.endswith("_here"):
        ren[c] = c + "_adj"                 # n_samples     -> n_samples_adj

rsrp_out = rsrp_cells.rename(columns=ren).copy()

# v2 carried rsrp_mean and rsrp_median. Rebuild them on the as-delivered basis,
# at full precision this time (v2 stored 1 dp, which cost 0.005 dB).
rsrp_out["rsrp_mean"]   = (rsrp_out["rsrp_sum"] / rsrp_out["n_samples"]).round(4)
rsrp_out = rsrp_out.rename(columns={"rsrp_median_here": "rsrp_median"})
rsrp_out = rsrp_out.drop(columns=[c for c in ("rsrp_mean_here_adj", "rsrp_mean_here",
                                              "rsrp_median_here_adj") if c in rsrp_out])

# v2's ten columns, in v2's order, so the file opens looking familiar.
# `cell_key` is added because a stable cell identifier makes every {FIXED} in the
# dashboard simpler than keying on two floats — v2 had to use lat+lon.
V2_FIRST = ["date", "hour", "RadioOperatorName", "rsrp_mean", "rsrp_median",
            "rsrp_min", "rsrp_max", "n_samples", "lat", "lon"]
BANDS_V5 = [f"n_{b}" for b in BAND_SHORT.values()]
rsrp_out = rsrp_out[V2_FIRST + ["bin_size", "cell_key", "rsrp_sum"] + BANDS_V5
                    + ["n_samples_adj", "rsrp_sum_adj", "rsrp_min_adj", "rsrp_max_adj"]
                    + [c + "_adj" for c in BANDS_V5]]

print("rsrp_cells.csv columns (v2 names first, additions after):")
print("  " + " | ".join(rsrp_out.columns))
assert set(V2_FIRST).issubset(rsrp_out.columns), "a v2 column name was lost"

rsrp_cells.csv columns (v2 names first, additions after):
  date | hour | RadioOperatorName | rsrp_mean | rsrp_median | rsrp_min | rsrp_max | n_samples | lat | lon | bin_size | cell_key | rsrp_sum | n_excellent | n_good | n_fair | n_poor | n_very_poor | n_samples_adj | rsrp_sum_adj | rsrp_min_adj | rsrp_max_adj | n_excellent_adj | n_good_adj | n_fair_adj | n_poor_adj | n_very_poor_adj


In [20]:
# ── traffic_bins: v2's tv_sum was ALREADY suppressed, so it keeps its name ───
traffic_out = traffic_bins.rename(columns={"tv_median_here": "tv_median"}).copy()
V2_T = ["date", "hour", "RadioOperatorName", "TrafficDirection",
        "tv_sum", "tv_median", "n_samples", "bin_size", "lat", "lon"]
traffic_out = traffic_out[V2_T + ["cell_key", "tv_sum_raw", "n_samples_raw", "tv_max", "tv_max_raw"]]
assert set(V2_T).issubset(traffic_out.columns), "a v2 column name was lost"
print("traffic_bins.csv columns:")
print("  " + " | ".join(traffic_out.columns))

traffic_bins.csv columns:
  date | hour | RadioOperatorName | TrafficDirection | tv_sum | tv_median | n_samples | bin_size | lat | lon | cell_key | tv_sum_raw | n_samples_raw | tv_max | tv_max_raw


In [21]:
# ── device_rsrp: device x operator x date, as v2 had it, plus p10 and the flag ─
dev_day = (rsrp.groupby(DGRP + ["date"], observed=True)
               .agg(n_samples=("RSRP", "size"), rsrp_sum=("RSRP", "sum"),
                    rsrp_avg=("RSRP", "mean"), rsrp_min=("RSRP", "min"),
                    rsrp_max=("RSRP", "max"), rsrp_p10=("RSRP", p10),
                    rsrp_p90=("RSRP", p90))
               .reset_index())
dev_day = dev_day[dev_day["n_samples"] > 0].merge(qflag, on=DGRP, how="left")
for c in ("rsrp_avg", "rsrp_p10", "rsrp_p90"):
    dev_day[c] = dev_day[c].round(3)
dev_day = dev_day[["DeviceName", "DeviceManufacturer", "RadioOperatorName", "date",
                   "rsrp_avg", "rsrp_min", "rsrp_max", "rsrp_p90",
                   "n_samples", "rsrp_sum", "rsrp_p10", "is_quantised"]]
assert dev_day["n_samples"].sum() == len(rsrp)
print(f"device_rsrp.csv : {len(dev_day):,} rows   (v2 had 939 — the drop is the alias collapse)")

device_rsrp.csv : 934 rows   (v2 had 939 — the drop is the alias collapse)


In [22]:
# ── operator_summary: v2's column names, v2's meanings, plus both toggle states ─
G = "RadioOperatorName"
s1 = rsrp_out[rsrp_out["bin_size"] == JOIN_BIN]
t1 = traffic_out[traffic_out["bin_size"] == JOIN_BIN]

rows_os = []
for op in sorted(rsrp[G].astype(str).unique()):
    r_ = s1[s1[G].astype(str) == op]
    t_ = t1[t1[G].astype(str) == op]
    v_ = val[val[G].astype(str) == op]

    def cells(nc, sc):
        c = r_.groupby("cell_key", observed=True).agg(n=(nc, "sum"), s=(sc, "sum"))
        c = c[c["n"] >= MIN_CELL_SAMPLES]
        m = c["s"] / c["n"]
        return len(c), int((m < WEAK_CELL_DBM).sum())

    n_cells,  n_weak  = cells("n_samples", "rsrp_sum")
    n_cellsa, n_weaka = cells("n_samples_adj", "rsrp_sum_adj")
    dl = t_[t_["TrafficDirection"].astype(str) == "Downlink"]
    ul = t_[t_["TrafficDirection"].astype(str) == "Uplink"]

    rows_os.append({
        G: op,
        "measurements":   int(r_["n_samples"].sum()),
        "mean_rsrp":      round(r_["rsrp_sum"].sum() / r_["n_samples"].sum(), 3),
        "cells":          n_cells, "weak_cells": n_weak,
        "weak_cell_pct":  round(100 * n_weak / n_cells, 2),
        "downlink_gb":    round(dl["tv_sum"].sum() / 1024, 2),
        "uplink_gb":      round(ul["tv_sum"].sum() / 1024, 2),
        "measurements_adj": int(r_["n_samples_adj"].sum()),
        "mean_rsrp_adj":  round(r_["rsrp_sum_adj"].sum() / r_["n_samples_adj"].sum(), 3),
        "cells_adj":      n_cellsa, "weak_cells_adj": n_weaka,
        "weak_cell_pct_adj": round(100 * n_weaka / n_cellsa, 2),
        "downlink_gb_raw": round(dl["tv_sum_raw"].sum() / 1024, 2),
        "uplink_gb_raw":   round(ul["tv_sum_raw"].sum() / 1024, 2),
        "pct_below_service":     round(100 * v_.loc[v_["RSRP"] < SERVICE_DBM, "n"].sum() / v_["n"].sum(), 2),
        "pct_below_service_adj": round(100 * v_.loc[~v_["is_quantised"] & (v_["RSRP"] < SERVICE_DBM), "n"].sum()
                                       / v_.loc[~v_["is_quantised"], "n"].sum(), 2),
        "n_devices":            int(dev_out.loc[dev_out[G].astype(str) == op, "DeviceName"].nunique()),
        "n_devices_quantised":  int(dev_out.loc[(dev_out[G].astype(str) == op) & dev_out["is_quantised"],
                                                "DeviceName"].nunique()),
        "quantised_sample_pct": round(100 * rsrp.loc[rsrp[G].astype(str) == op, "is_quantised"].mean(), 2),
    })

op_sum = pd.DataFrame(rows_os)
for col, base in [("measurements", "measurement_share"), ("downlink_gb", "downlink_share")]:
    op_sum[base] = (100 * op_sum[col] / op_sum[col].sum()).round(2)
op_sum["rsrp_rank"]     = op_sum["mean_rsrp"].rank(ascending=False).astype(int)
op_sum["rsrp_rank_adj"] = op_sum["mean_rsrp_adj"].rank(ascending=False).astype(int)

tm_path = PROCESSED_DIR / "tech_mix.csv"
if tm_path.exists():
    tm = pd.read_csv(tm_path)
    op_sum["lte_attach_pct"] = op_sum[G].map(
        tm[tm["RadioNetworkGeneration"] == "4G"].set_index(G)["pct_of_operator"])

display(op_sum[[G, "measurements", "mean_rsrp", "mean_rsrp_adj", "rsrp_rank",
                "rsrp_rank_adj", "downlink_gb", "downlink_gb_raw"]])

# The v2 file's own numbers, as a regression test against the delivered extract.
V2_KNOWN = {"Operator A": (1079055, -85.47, 631.7),
            "Operator B": (445983,  -84.65, 452.3),
            "Operator C": (450534,  -80.87, 321.5)}
for op, (m, mr, dl_) in V2_KNOWN.items():
    row = op_sum[op_sum[G] == op].iloc[0]
    assert row["measurements"] == m,                f"{op} measurements moved: {row['measurements']} vs {m}"
    assert abs(row["mean_rsrp"] - mr) < 0.006,      f"{op} mean_rsrp moved"
    assert abs(row["downlink_gb"] - dl_) < 0.05,    f"{op} downlink moved"
print("\nMatches v2's operator_summary.csv exactly on measurements, mean RSRP and downlink.")
print("Anything built on those numbers in the existing workbook will not move.")

,RadioOperatorName,measurements,mean_rsrp,mean_rsrp_adj,rsrp_rank,rsrp_rank_adj,downlink_gb,downlink_gb_raw
0,Operator A,1079055,-85.473,-83.561,3,2,631.73,631.73
1,Operator B,445983,-84.647,-84.789,2,3,452.30,1853.85
2,Operator C,450534,-80.875,-80.828,1,1,321.46,321.46



Matches v2's operator_summary.csv exactly on measurements, mean RSRP and downlink.
Anything built on those numbers in the existing workbook will not move.


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# Write the nine files
# ─────────────────────────────────────────────────────────────────────────────
register("rsrp_cells.csv",             rsrp_out,    "bin_size x date x hour x operator x cell")
register("traffic_bins.csv",           traffic_out, "bin_size x date x hour x operator x direction x cell")
register("device_operator_totals.csv", dev_out,     "device x operator")
register("device_rsrp.csv",            dev_day,     "device x operator x date")
register("operator_summary.csv",       op_sum,      "operator (all dates - do not filter)")
register("rsrp_value_counts.csv",      val,         "date x hour x operator x is_quantised x RSRP value")
register("device_value_counts.csv",    dval,        "device x operator x RSRP value")

Written: rsrp_cells.csv                 87,802 rows x 27 cols    10.08 MB


Written: traffic_bins.csv              146,284 rows x 15 cols    14.49 MB
Written: device_operator_totals.csv        396 rows x 23 cols     0.05 MB
Written: device_rsrp.csv                   934 rows x 12 cols     0.08 MB
Written: operator_summary.csv                3 rows x 25 cols     0.00 MB
Written: rsrp_value_counts.csv          16,786 rows x  6 cols     0.60 MB
Written: device_value_counts.csv        14,403 rows x  6 cols     0.59 MB


PosixPath('/root/v5/Data/processed/device_value_counts.csv')

---
## Section 7 — `narrative.xlsx`  ·  the four evidence panels

Four dashboard panels are not aggregations of the measurement data — they are the *account*
of what was done to it. The validation funnel, the duplicate-alias panel, the residual-issues
list and the closing takeaways.

In the first draft the guide told you to build two of these by hand in Excel. That is a bad
instruction: a hand-made table drifts from the pipeline the moment a threshold changes, and
nobody notices. They are generated here instead, from the same run that produced everything
else, so the funnel can never disagree with the cleaning it describes.

In [24]:
# ── cleaning_log: both datasets in one table, so the funnel can show either ──
logs = []
for name, fn in (("RSRP", "rsrp_cleaning_log.csv"), ("Traffic", "traffic_cleaning_log.csv")):
    fp = REF_DIR / fn
    if fp.exists():
        lg = pd.read_csv(fp)
        lg.insert(0, "dataset", name)
        lg["step_order"] = range(1, len(lg) + 1)
        logs.append(lg)
cleaning_log_all = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()

# ── device_aliases ───────────────────────────────────────────────────────────
alias_fp = REF_DIR / "device_alias_map.csv"
device_aliases = pd.read_csv(alias_fp) if alias_fp.exists() else pd.DataFrame()

print(f"cleaning_log   : {len(cleaning_log_all)} steps across "
      f"{cleaning_log_all['dataset'].nunique() if len(cleaning_log_all) else 0} datasets")
print(f"device_aliases : {len(device_aliases)} alias groups")

cleaning_log   : 16 steps across 2 datasets
device_aliases : 28 alias groups


In [25]:
# ── residual_issues: what was documented rather than silently fixed ─────────
n_quant_pairs = int(dev_out["is_quantised"].sum())
quant_rows    = int(rsrp["is_quantised"].sum())
odo_rows      = int(traffic["is_odometer"].sum())
n_naive       = int(dev_out.loc[dev_out["n_operators_solid"] >= 2, "DeviceName"].nunique())
n_matched     = int(dev_out.loc[dev_out["comparable"], "DeviceName"].nunique())
top10_share   = round(100 * dev_out.groupby("DeviceName")["n_samples"].sum()
                      .nlargest(10).sum() / dev_out["n_samples"].sum(), 1)

residual_issues = pd.DataFrame([
 dict(severity="Documented", issue=f"{n_quant_pairs} quantised model/operator pairs retained",
      detail=f"{quant_rows:,} rows ({100*quant_rows/len(rsrp):.1f}%). Flagged, never deleted — "
             f"the correction toggle switches them out."),
 dict(severity="Documented", issue=f"{odo_rows} cumulative-counter traffic rows retained",
      detail="Flagged, never deleted. The suppression toggle switches them out."),
 dict(severity="Structural", issue="No subscriber identifier in either source file",
      detail="Every mention of 'users' means measurement samples. Cannot be fixed by cleaning."),
 dict(severity="Structural", issue=f"Ten handset models are {top10_share}% of all samples",
      detail="Device readings are indicative of those handsets in those places, not fleet-wide."),
 dict(severity="Documented", issue=f"{n_naive - n_matched} of {n_naive} comparable models share no cell",
      detail=f"Only {n_matched} models have a location-matched cross-operator comparison. "
             f"The `comparable` flag marks them."),
 dict(severity="Documented", issue="The coverage-traffic correlation depends on the grid",
      detail="r = +0.36 at 2 km, near zero at 500 m. Always quote the grid beside the number."),
 dict(severity="Resolved", issue="Friday 1 Nov dropped (64 rows)",
      detail="Timezone-conversion residue, not an observation day."),
 dict(severity="Resolved", issue="Device aliases unified before deduplication",
      detail=f"{len(device_aliases)} alias groups collapsed, so each handset appears once."),
])
display(residual_issues)

,severity,issue,detail
0,Documented,6 quantised model/operator pairs retained,"273,112 rows (13.8%). Flagged, never deleted —..."
1,Documented,14 cumulative-counter traffic rows retained,"Flagged, never deleted. The suppression toggle..."
2,Structural,No subscriber identifier in either source file,Every mention of 'users' means measurement sam...
3,Structural,Ten handset models are 60.0% of all samples,Device readings are indicative of those handse...
4,Documented,36 of 65 comparable models share no cell,Only 29 models have a location-matched cross-o...
5,Documented,The coverage-traffic correlation depends on th...,"r = +0.36 at 2 km, near zero at 500 m. Always ..."
6,Resolved,Friday 1 Nov dropped (64 rows),"Timezone-conversion residue, not an observatio..."
7,Resolved,Device aliases unified before deduplication,"28 alias groups collapsed, so each handset app..."


In [26]:
# ── takeaways: three per operator, with the numbers filled in from this run ──
rows_tk = []
s1 = rsrp_cells[rsrp_cells["bin_size"] == JOIN_BIN]
t1 = traffic_bins[traffic_bins["bin_size"] == JOIN_BIN]

mean_adj = (s1.groupby("RadioOperatorName", observed=True)
              .apply(lambda g: g["rsrp_sum"].sum() / g["n_samples"].sum(), include_groups=False))
rank_adj = mean_adj.rank(ascending=False).astype(int)
mean_raw = (s1.groupby("RadioOperatorName", observed=True)
              .apply(lambda g: g["rsrp_sum_raw"].sum() / g["n_samples_raw"].sum(), include_groups=False))
rank_raw = mean_raw.rank(ascending=False).astype(int)

cw = s1.groupby(["cell_key", "RadioOperatorName"], observed=True).agg(
        n=("n_samples_raw", "sum"), s=("rsrp_sum_raw", "sum")).reset_index()
cw = cw[cw["n"] >= 25]; cw["m"] = cw["s"] / cw["n"]
won = cw[cw["m"] == cw.groupby("cell_key")["m"].transform("max")]
scored = cw["cell_key"].nunique()

for op in sorted(mean_adj.index.astype(str)):
    fl = 100 * rsrp.loc[rsrp["RadioOperatorName"].astype(str) == op, "is_quantised"].mean()
    d = t1[(t1["RadioOperatorName"].astype(str) == op) &
           (t1["TrafficDirection"].astype(str) == "Downlink")]
    dl_c, dl_r = d["tv_sum"].sum() / 1024, d["tv_sum_raw"].sum() / 1024
    n_won = int((won["RadioOperatorName"].astype(str) == op).sum())
    best = mean_adj.max()

    rows_tk += [
      dict(RadioOperatorName=op, rank=1,
           headline="Fix the measurement before you fix the network"
                    if fl > 5 else "Your measurement base is clean",
           body=(f"{fl:.1f}% of this network's samples come from handsets that report a lookup "
                 f"table rather than a measurement. Correcting for it moves the network from "
                 f"rank {rank_raw[op]} to rank {rank_adj[op]} on mean RSRP."
                 if fl > 5 else
                 f"Only {fl:.1f}% of samples come from quantised handsets, so the corrected and "
                 f"as-delivered rankings agree (rank {rank_adj[op]} either way).")),
      dict(RadioOperatorName=op, rank=2,
           headline="Footprint versus depth",
           body=(f"This network leads {n_won} of {scored} scored {JOIN_BIN} cells. Its corrected "
                 f"mean RSRP is {mean_adj[op]:.2f} dBm, "
                 + ("the best of the three." if abs(mean_adj[op] - best) < 1e-9
                    else f"{abs(mean_adj[op] - best):.1f} dB behind the leader."))),
      dict(RadioOperatorName=op, rank=3,
           headline="Traffic position",
           body=(f"Downlink carried is {dl_c:,.1f} GB on suppressed samples"
                 + (f", against {dl_r:,.1f} GB if the {odo_rows} counter rows are left in — "
                    f"a {dl_r/dl_c:.1f}x overstatement."
                    if dl_r / max(dl_c, 1e-9) > 1.05 else
                    ". The implausible-sample toggle does not move this network at all."))),
    ]

takeaways = pd.DataFrame(rows_tk)
display(takeaways[takeaways["RadioOperatorName"] == takeaways["RadioOperatorName"].iloc[0]])

,RadioOperatorName,rank,headline,body
0,Operator A,1,Fix the measurement before you fix the network,23.9% of this network's samples come from hand...
1,Operator A,2,Footprint versus depth,This network leads 574 of 1361 scored 1km cell...
2,Operator A,3,Traffic position,Downlink carried is 631.7 GB on suppressed sam...


In [27]:
NARRATIVE_PATH = PROCESSED_DIR / "narrative.xlsx"
with pd.ExcelWriter(NARRATIVE_PATH, engine="openpyxl") as xl:
    cleaning_log_all.to_excel(xl, sheet_name="cleaning_log",    index=False)
    device_aliases.to_excel(xl,   sheet_name="device_aliases",  index=False)
    residual_issues.to_excel(xl,  sheet_name="residual_issues", index=False)
    takeaways.to_excel(xl,        sheet_name="takeaways",       index=False)

mb = NARRATIVE_PATH.stat().st_size / 1024**2
manifest.append({"file": "narrative.xlsx",
                 "rows": len(cleaning_log_all) + len(device_aliases)
                         + len(residual_issues) + len(takeaways),
                 "cols": 4, "MB": round(mb, 2),
                 "grain": "4 sheets: cleaning_log, device_aliases, residual_issues, takeaways"})
print(f"Written: narrative.xlsx  4 sheets, {mb:.2f} MB")

Written: narrative.xlsx  4 sheets, 0.01 MB


---
## Manifest and final verification

The last cell re-reads every file **from disk** — not from the frames still in memory — and
re-runs the reconciliation. Reading back is the point: it catches anything that goes wrong in
the write itself, which is a real failure mode (a float silently truncated by rounding, a
`NaN` written as an empty field and read back as a string, a column reordered).

In [28]:
man = pd.DataFrame(manifest)
tech = PROCESSED_DIR / "tech_mix.csv"
if tech.exists():
    tm = pd.read_csv(tech)
    man = pd.concat([man, pd.DataFrame([{
        "file": "tech_mix.csv", "rows": len(tm), "cols": tm.shape[1],
        "MB": round(tech.stat().st_size / 1024**2, 2),
        "grain": "operator x generation (written by Notebook 01)"}])], ignore_index=True)

man = man.sort_values("file").reset_index(drop=True)
print("EXTRACT MANIFEST")
display(man)
print(f"\nTotal: {len(man)} files, {man['rows'].sum():,} rows, {man['MB'].sum():.1f} MB")

EXTRACT MANIFEST


,file,rows,cols,MB,grain
0,device_operator_totals.csv,396,23,0.05,device x operator
1,device_rsrp.csv,934,12,0.08,device x operator x date
2,device_value_counts.csv,14403,6,0.59,device x operator x RSRP value
3,narrative.xlsx,61,4,0.01,"4 sheets: cleaning_log, device_aliases, residu..."
4,operator_summary.csv,3,25,0.00,operator (all dates - do not filter)
5,rsrp_cells.csv,87802,27,10.08,bin_size x date x hour x operator x cell
6,rsrp_value_counts.csv,16786,6,0.60,date x hour x operator x is_quantised x RSRP v...
7,tech_mix.csv,15,4,0.00,operator x generation (written by Notebook 01)
8,traffic_bins.csv,146284,15,14.49,bin_size x date x hour x operator x direction ...



Total: 9 files, 266,684 rows, 25.9 MB


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# Read back from disk and verify
# ─────────────────────────────────────────────────────────────────────────────
back = {f: pd.read_csv(PROCESSED_DIR / f) for f in man["file"] if f.endswith(".csv")}
back["takeaways"] = pd.read_excel(PROCESSED_DIR / "narrative.xlsx", sheet_name="takeaways")

N_SRC      = len(rsrp)
N_SRC_ADJ  = int((~rsrp["is_quantised"]).sum())
MEAN_SRC   = rsrp["RSRP"].mean()
MEAN_ADJ   = rsrp.loc[~rsrp["is_quantised"], "RSRP"].mean()
TV_SRC     = traffic["TrafficVolume"].sum()
TV_CLN     = traffic.loc[~traffic["is_odometer"], "TrafficVolume"].sum()

checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

# rsrp_cells — per bin size, both states
rc = back["rsrp_cells.csv"]
for _, label in BIN_SIZES:
    s = rc[rc["bin_size"] == label]
    check(f"rsrp_cells[{label}] as-delivered sample count",
          s["n_samples"].sum() == N_SRC, f"{s['n_samples'].sum():,}")
    check(f"rsrp_cells[{label}] corrected sample count",
          s["n_samples_adj"].sum() == N_SRC_ADJ, f"{s['n_samples_adj'].sum():,}")
    check(f"rsrp_cells[{label}] as-delivered weighted mean",
          abs(s["rsrp_sum"].sum()/s["n_samples"].sum() - MEAN_SRC) < 1e-6,
          f"{s['rsrp_sum'].sum()/s['n_samples'].sum():.4f}")
    check(f"rsrp_cells[{label}] corrected weighted mean",
          abs(s["rsrp_sum_adj"].sum()/s["n_samples_adj"].sum() - MEAN_ADJ) < 1e-6,
          f"{s['rsrp_sum_adj'].sum()/s['n_samples_adj'].sum():.4f}")
    check(f"rsrp_cells[{label}] band counts sum to n_samples",
          (s[[f"n_{b}" for b in BAND_SHORT.values()]].sum(axis=1) == s["n_samples"]).all())

# traffic_bins
tb = back["traffic_bins.csv"]
for _, label in BIN_SIZES:
    s = tb[tb["bin_size"] == label]
    check(f"traffic_bins[{label}] raw volume",
          abs(s["tv_sum_raw"].sum() - TV_SRC) < 1e-2, f"{s['tv_sum_raw'].sum()/1024:,.1f} GB")
    check(f"traffic_bins[{label}] clean volume",
          abs(s["tv_sum"].sum() - TV_CLN) < 1e-2, f"{s['tv_sum'].sum()/1024:,.1f} GB")

# device tables
check("device_operator_totals sample count",
      back["device_operator_totals.csv"]["n_samples"].sum() == N_SRC)
check("device_rsrp sample count",
      back["device_rsrp.csv"]["n_samples"].sum() == N_SRC)
osum = back["operator_summary.csv"]
check("operator_summary matches v2 measurements",
      osum["measurements"].sum() == N_SRC, f"{osum['measurements'].sum():,}")
check("takeaways cover every operator", back["takeaways"]["RadioOperatorName"].nunique() == 3)

# value counts — the strongest test: the full distribution must be intact
v = back["rsrp_value_counts.csv"]
check("rsrp_value_counts total", v["n"].sum() == N_SRC, f"{v['n'].sum():,}")
check("rsrp_value_counts recovers the mean",
      abs((v["RSRP"] * v["n"]).sum() / v["n"].sum() - MEAN_SRC) < 1e-9)
va = v[~v["is_quantised"]]
check("rsrp_value_counts recovers the corrected mean",
      abs((va["RSRP"] * va["n"]).sum() / va["n"].sum() - MEAN_ADJ) < 1e-9)
d = back["device_value_counts.csv"]
check("device_value_counts total", d["n"].sum() == N_SRC, f"{d['n'].sum():,}")

# The model must reproduce, from disk, the headline numbers the retired
# operator_summary.csv used to serve.
o = back["rsrp_cells.csv"]
o1 = o[o["bin_size"] == JOIN_BIN]
opA = o1[o1["RadioOperatorName"] == "Operator A"]
check("Operator A mean RSRP, as delivered (= v2)",
      abs(opA["rsrp_sum"].sum() / opA["n_samples"].sum() + 85.473) < 0.001,
      f"{opA['rsrp_sum'].sum() / opA['n_samples'].sum():.3f}")
check("Operator A mean RSRP, corrected",
      abs(opA["rsrp_sum_adj"].sum() / opA["n_samples_adj"].sum() + 83.561) < 0.001,
      f"{opA['rsrp_sum_adj'].sum() / opA['n_samples_adj'].sum():.3f}")

tbk = back["traffic_bins.csv"]
t1  = tbk[(tbk["bin_size"] == JOIN_BIN) & (tbk["TrafficDirection"] == "Downlink")]
check("Operator B downlink, suppressed",
      abs(t1.loc[t1["RadioOperatorName"] == "Operator B", "tv_sum"].sum() / 1024 - 452.30) < 0.02)
check("Operator B downlink, all samples",
      abs(t1.loc[t1["RadioOperatorName"] == "Operator B", "tv_sum_raw"].sum() / 1024 - 1853.85) < 0.02)
check("total downlink reconciles with the source",
      abs(t1["tv_sum_raw"].sum()
          - traffic.loc[traffic["TrafficDirection"].astype(str) == "Downlink",
                        "TrafficVolume"].sum()) < 1e-2)

# tech_mix
if "tech_mix.csv" in back:
    check("tech_mix operator percentages sum to 100",
          back["tech_mix.csv"].groupby("RadioOperatorName")["pct_of_operator"]
              .sum().sub(100).abs().max() < 0.05)

res = pd.DataFrame(checks)
display(res)

n_fail = int((res["result"] == "FAIL").sum())
print(f"\n{len(res) - n_fail}/{len(res)} checks passed.")
assert n_fail == 0, f"{n_fail} verification checks FAILED — do not use these extracts"
print("\nAll extracts verified against the source data, read back from disk.")

,check,result,detail
0,rsrp_cells[500m] as-delivered sample count,PASS,"1,975,572"
1,rsrp_cells[500m] corrected sample count,PASS,"1,702,460"
2,rsrp_cells[500m] as-delivered weighted mean,PASS,-84.2380
3,rsrp_cells[500m] corrected weighted mean,PASS,-83.1587
4,rsrp_cells[500m] band counts sum to n_samples,PASS,
5,rsrp_cells[1km] as-delivered sample count,PASS,"1,975,572"
6,rsrp_cells[1km] corrected sample count,PASS,"1,702,460"
7,rsrp_cells[1km] as-delivered weighted mean,PASS,-84.2380
8,rsrp_cells[1km] corrected weighted mean,PASS,-83.1587
9,rsrp_cells[1km] band counts sum to n_samples,PASS,



35/35 checks passed.

All extracts verified against the source data, read back from disk.


---
## How to use these in Tableau

**Load all eight as separate data sources.** Do not blend them in the connection pane —
they are different grains, and Tableau's default join would multiply rows. Where two are
needed together (a coverage-vs-traffic scatter), join `rsrp_cells` to `traffic_bins` on
`cell_key` **and** `bin_size`, after filtering both to one bin size.

**Every average is a calculated field, never `AVG()`.**

```
Mean RSRP  =  SUM([rsrp_sum]) / SUM([n_samples])
```

`AVG([rsrp_mean_here])` gives a different — and wrong — number the moment more than one row
is in scope. The `_here` suffix is there to make that visible at the point of use.

**The two toggles are one parameter each**, switching which column a calculation reads:

```
Correct handset artefacts  →  [Apply correction] = True ? SUM([rsrp_sum])  : SUM([rsrp_sum_raw])
                              [Apply correction] = True ? SUM([n_samples]) : SUM([n_samples_raw])

Suppress implausible traffic → [Suppress] = True ? SUM([tv_sum]) : SUM([tv_sum_raw])
```

**Band shares recompute, they do not average:**

```
% Excellent  =  SUM([n_excellent]) / SUM([n_samples])
```

**Percentiles come from `rsrp_value_counts`,** using a running sum of `n` ordered by `RSRP`
— the value where the running share crosses 0.5 is the exact median at whatever filter is
applied. No approximation, no sampling.

---

## What each file is for

| File | Use it for | Do not use it for |
|---|---|---|
| `fact_rsrp_cells.csv` | maps, coverage by hour/day, band mix, weak-cell counts | handset questions |
| `fact_traffic_bins.csv` | traffic maps, hourly profiles, the coverage↔traffic join | anything handset-level |
| `fact_device_totals.csv` | handset ranking and the gap table | anything by date — no fact carries device × date |
| `fact_rsrp_values.csv` | distributions, exact percentiles, box plots | spatial questions |
| `fact_device_values.csv` | the quantisation evidence panel | ranking — use the totals file |
| `fact_tech_mix.csv` | LTE attach rate, generation mix | anything post-scoping |

### One caution worth repeating

`gap_vs_best` and `matched_gap_vs_best` will disagree, sometimes by a lot. When they do,
**the matched column is the one that says something about the network** — the naive column
is comparing different places at different times. Any caption that makes a claim about
operator performance from a handset chart should be reading the matched column and the
`comparable` flag.